## Cassegrain Telescope ##

In [ ]:
import copy

import numpy as np

from prtlab import (
    example_cassegrain_telescope,
    plot_jones_pupil,
    plot_prt_lens_cross_section,
    plot_prt_ray_trace,
    polarization_ray_trace,
    select_dominant_final_ray,
    transform_p_to_jones,
    update_clear_apertures_from_ray_trace,
)


Here we do some analysis of a Cassegrain System to Exercise some of the features of prtLab.  First let's load this example, which consists of two mirrors.

In [ ]:
T = example_cassegrain_telescope()
T


The mirrors are not odd apsheres, but they do have conics so the iterative ray tracing to find the surface intersection points works.  

Here is a simple plot of the system.

In [ ]:
k0 = np.array([0, 0, 1])
x0 = np.array([0, 4000, -6000])
x1 = np.array([0, -4000, -6000])
options = {"minAmplitude": 0.05}
d1 = polarization_ray_trace(T, k0, x0, [1, 0], options)
d2 = polarization_ray_trace(T, k0, x1, [1, 0], options)
Tplot = copy.deepcopy(T)
update_clear_apertures_from_ray_trace(Tplot, [d1, d2])
axis, _ = plot_prt_lens_cross_section(Tplot)
_ = plot_prt_ray_trace([d1, d2], ax=axis)


For this system, we want to compute the Jones Pupils for this system to see the impact of polarization aberrations due to reflection from the Aluminum mirrors.



In [ ]:
d = 4161.65
pupil_axis = np.linspace(-d, d, 50)
X, Y = np.meshgrid(pupil_axis, pupil_axis)
rho = np.hypot(X, Y)
ftr = (rho >= 1000) & (rho <= d)
E_in = np.array([1, 0])
k_in = np.array([0, 0, 1])
Jall = np.full((*X.shape, 2, 2), np.nan + 0j)
OPL = np.full(X.shape, np.nan)
coordinate = {
    "type": "doublePole", "a_loc": k_in,
    "x_o": np.array([1, 0, 0]),
}

def trace_pupil(system):
    jones = np.full((*X.shape, 2, 2), np.nan + 0j)
    opl = np.full(X.shape, np.nan)
    for index in np.ndindex(X.shape):
        ray_output = polarization_ray_trace(
            system, k_in, [X[index], Y[index], -6000], E_in
        )
        if ray_output.final_ray_ids:
            final_ray, _ = select_dominant_final_ray(ray_output)
            jones[index] = transform_p_to_jones(
                final_ray.P, k_in, final_ray.k, coordinate
            )
            opl[index] = final_ray.opl
    return jones, opl

Jall, OPL = trace_pupil(T)
_ = plot_jones_pupil(Jall, ftr)


Here the machinery of the polarization transfer matrices show the impact of angle dependent Fresnel reflection on the transfer function of the system.  Of particular note is the astigmatism seen in the off diagonal terms.

There are a few ways to study this.  One way would be to make the mirrors perfect.  We can approximate this by making the mirrors index of refracation approach infinity: 

In [ ]:
T.surfaces[1].index_data["n"] = 1e15
T.surfaces[2].index_data["n"] = 1e15
Jall, _ = trace_pupil(T)
_ = plot_jones_pupil(Jall, ftr)


You can see that with a perfect mirror, the imaginary parts of the Jones terms are esssentially 0, and the diagonal terms are essentially 1.  So there are no polarization aberrations, "proving" that this behavior is caused by the metal reflection.  

You might ask why the imaginary part is identially 0. Aren't there OPD differences across the pupil?  The answer is yes, but the P matrices do not have this information encoded.  You could add it, but due to phase wrapping the result will not be useful.  It is possible to look at the OPD separately, by extracing the final ray OPD for each launched ray. 

Another way to check this is to use a different metal.  Let's try gold:

In [ ]:
n_gold = 0.8 + 1.8j
T.surfaces[1].index_data["n"] = n_gold
T.surfaces[2].index_data["n"] = n_gold
Jall, _ = trace_pupil(T)
_ = plot_jones_pupil(Jall, ftr)


You can see that the magnitude of the phase error is a function of the mirror material, which is further evidence that this behavior is dominated by Fresnel reflection induced aberrations.

You can do other calcs with this, such as looking at Maltese Cross Pattern vs Incident Polarization.  But for now I will stop here.

